# SpeculateForge GRPO Demo

This notebook is the Phase F scaffold for the required Colab training demo. Replace the placeholder Space URLs before sharing with judges.

In [ ]:
!pip install -q trl transformers trackio openenv-core requests datasets

In [ ]:
import json
import re
import requests
import trackio
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from training.callbacks import TrackioCallback
from training.trackio_integration import manual_step_metrics

ENV_URL = "https://YOUR_SPACE.hf.space"

trackio.init(
    project="speculate-forge-grpo",
    space_id="YOUR_USERNAME/speculate-forge-tracking",
    config={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "task": "task1_easy_a100",
        "bf16": True,
        "lr": 1e-5,
    },
)

def parse_config_from_completion(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {
        "num_speculative_tokens": 4,
        "acceptance_threshold": 0.5,
        "tree_depth": 1,
        "ngram_cache_size": 0,
        "adaptive_depth": False,
        "label": "default",
    }

def speculation_reward(completions, prompts=None, **kwargs):
    rewards = []
    for completion in completions:
        try:
            config = parse_config_from_completion(completion)
            response = requests.post(
                f"{ENV_URL}/manual_step",
                json=config,
                params={"task_level": 1},
                timeout=60,
            )
            payload = response.json()
            reward = float(payload.get("reward", 0.0))
            trackio.log(manual_step_metrics(payload))
        except Exception:
            reward = 0.0
        rewards.append(reward)
    return rewards


In [ ]:
SYSTEM = """You are an expert in LLM inference optimization.
Propose speculative decoding configs that maximize throughput while keeping quality >= 95%."""

dataset = Dataset.from_list([
    {"prompt": SYSTEM + "\n\nPropose an optimized speculative decoding config:"}
] * 200)

training_args = GRPOConfig(
    bf16=True,
    learning_rate=1e-5,
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=200,
    max_completion_length=256,
    logging_steps=5,
    output_dir="./speculate-forge-grpo",
    report_to="none",
)

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=training_args,
    reward_funcs=speculation_reward,
    train_dataset=dataset,
)
trainer.add_callback(TrackioCallback())
trainer.train()
trackio.finish()